# S10 - Unidad 1 - Evidencia de Aprendizaje 1 - Carga y Creación de Dataset

Este Notebook es el **orquestador** de la actividad:

```
Hugging Face → HFDatasetLoader → DataFrame → 4 transformaciones → df_final → DatasetSummarizer → resumen_dataset.txt
```

La lógica vive en las clases de `src/`; aquí solo se coordinan.

## 1. Importación de las clases

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.hf_dataset_loader import HFDatasetLoader
from src.dataset_summarizer import DatasetSummarizer

pd.set_option("display.max_colwidth", 80)
print("Clases importadas:", HFDatasetLoader.__name__, "y", DatasetSummarizer.__name__)

Clases importadas: HFDatasetLoader y DatasetSummarizer


## 2. Carga del dataset desde Hugging Face

Dataset: [`manuelgscom/dale-bash-test-v2`](https://huggingface.co/datasets/manuelgscom/dale-bash-test-v2).
Contiene peticiones en lenguaje natural (español e inglés) dirigidas a un asistente de terminal
Linux, junto con el tipo de respuesta esperada y los comandos bash de referencia.

El dataset solo publica el split **`test`**, así que se indica explícitamente con `split="test"`.

In [2]:
DATASET_ID = "manuelgscom/dale-bash-test-v2"

loader = HFDatasetLoader(DATASET_ID, split="test")
df = loader.load()

print("Dataset cargado:", DATASET_ID)
print("Tipo de objeto:", type(df))
print("Dimensiones (filas, columnas):", df.shape)

Dataset cargado: manuelgscom/dale-bash-test-v2
Tipo de objeto: <class 'pandas.DataFrame'>
Dimensiones (filas, columnas): (471, 8)


## 3. Información inicial del DataFrame original

In [3]:
df.head()

,id,lang,category,context,request,expected_type,reference_commands,notes
0,dev3-ficheros-es-001,es,ficheros,"{'os': 'linux', 'distro': 'ubuntu-22.04', 'pm': 'apt', 'tools': ['batcat', '...","Enséñame en forma de árbol todos los directorios de /srv/musica, incluidos l...",command,[tree -a -d /srv/musica],"[contexto] Debe usar tree para mostrar recursivamente solo directorios, incl..."
1,dev3-ficheros-es-002,es,ficheros,"{'os': 'linux', 'distro': 'ubuntu-24.04', 'pm': 'apt', 'tools': []}",Necesito crear /home/ana/Proyecto Sur/documentos y /home/ana/Proyecto Sur/im...,command,[mkdir -p -- '/home/ana/Proyecto Sur/documentos' '/home/ana/Proyecto Sur/ima...,"Debe crear ambos directorios y los padres necesarios, respetando el espacio ..."
2,dev3-ficheros-es-003,es,ficheros,"{'os': 'linux', 'distro': 'debian-12', 'pm': 'apt', 'tools': []}",Usa rsync para copiar /home/ines/borradores/contrato.odt a /mnt/respaldo/con...,command,[cp -p -- /home/ines/borradores/contrato.odt /mnt/respaldo/contrato.odt],[contexto][ausente] Falta rsync; cp -p permite copiar el fichero conservando...
3,dev3-ficheros-es-004,es,ficheros,"{'os': 'linux', 'distro': 'ubuntu-22.04', 'pm': 'apt', 'tools': ['sudo']}",Borra definitivamente todos los ficheros regulares cuyo nombre termine en .b...,command,[sudo find /srv/contabilidad -type f -name '*.bak' -delete],[peligroso] Debe borrar recursivamente solo ficheros regulares terminados en...
4,dev3-ficheros-es-005,es,ficheros,"{'os': 'linux', 'distro': 'ubuntu-24.04', 'pm': 'apt', 'tools': ['curl', 'gi...","Lista las entradas de /srv/entregas, sin incluir las ocultas, de modificació...",command,[ls -lt -- /srv/entregas],Debe mostrar el listado detallado ordenado por fecha de modificación descend...


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 471 entries, 0 to 470
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   id                  471 non-null    str   
 1   lang                471 non-null    str   
 2   category            471 non-null    str   
 3   context             471 non-null    object
 4   request             471 non-null    str   
 5   expected_type       471 non-null    str   
 6   reference_commands  471 non-null    object
 7   notes               471 non-null    str   
dtypes: object(2), str(6)
memory usage: 140.1+ KB


In [5]:
df.isnull().sum()

id                    0
lang                  0
category              0
context               0
request               0
expected_type         0
reference_commands    0
notes                 0
dtype: int64

In [6]:
# La columna context es un diccionario anidado
df["context"].iloc[0]

{'os': 'linux',
 'distro': 'ubuntu-22.04',
 'pm': 'apt',
 'tools': array(['batcat', 'fdfind', 'rg', 'tree'], dtype=object)}

## 4. Transformaciones

### Transformación 1 — Expandir la columna anidada `context`

`context` guarda en un diccionario el entorno de cada petición: sistema operativo, distribución,
gestor de paquetes y herramientas instaladas. Con `pd.json_normalize` cada clave pasa a ser una
**columna propia** (`os`, `distro`, `pm`, `tools`), lo que permite filtrar y resumir por ellas.

In [7]:
context_columns = pd.json_normalize(df["context"].tolist())
df_expanded = pd.concat([df.drop(columns="context"), context_columns], axis=1)

print("Columnas nuevas:", list(context_columns.columns))
print("Dimensiones:", df.shape, "->", df_expanded.shape)
df_expanded[["id", "os", "distro", "pm", "tools"]].head()

Columnas nuevas: ['os', 'distro', 'pm', 'tools']
Dimensiones: (471, 8) -> (471, 11)


,id,os,distro,pm,tools
0,dev3-ficheros-es-001,linux,ubuntu-22.04,apt,"[batcat, fdfind, rg, tree]"
1,dev3-ficheros-es-002,linux,ubuntu-24.04,apt,[]
2,dev3-ficheros-es-003,linux,debian-12,apt,[]
3,dev3-ficheros-es-004,linux,ubuntu-22.04,apt,[sudo]
4,dev3-ficheros-es-005,linux,ubuntu-24.04,apt,"[curl, git, jq, ssh]"


### Transformación 2 — Seleccionar columnas relevantes

Se descartan columnas que no aportan al análisis:

- `os`: vale `linux` en todas las filas, así que no distingue nada.
- `notes`: comentario libre para quien evalúa las respuestas, no describe la petición.

Se conservan el identificador, el idioma, la categoría, el entorno, la petición, el tipo de
respuesta esperado y los comandos de referencia.

In [8]:
print("Valores distintos de os:", df_expanded["os"].unique())

selected_columns = ["id", "lang", "category", "distro", "pm", "tools",
                    "request", "expected_type", "reference_commands"]
df_selected = df_expanded[selected_columns].copy()

print("Columnas seleccionadas:", list(df_selected.columns))
print("Dimensiones:", df_selected.shape)
df_selected.head()

Valores distintos de os: <ArrowStringArray>
['linux']
Length: 1, dtype: str
Columnas seleccionadas: ['id', 'lang', 'category', 'distro', 'pm', 'tools', 'request', 'expected_type', 'reference_commands']
Dimensiones: (471, 9)


,id,lang,category,distro,pm,tools,request,expected_type,reference_commands
0,dev3-ficheros-es-001,es,ficheros,ubuntu-22.04,apt,"[batcat, fdfind, rg, tree]","Enséñame en forma de árbol todos los directorios de /srv/musica, incluidos l...",command,[tree -a -d /srv/musica]
1,dev3-ficheros-es-002,es,ficheros,ubuntu-24.04,apt,[],Necesito crear /home/ana/Proyecto Sur/documentos y /home/ana/Proyecto Sur/im...,command,[mkdir -p -- '/home/ana/Proyecto Sur/documentos' '/home/ana/Proyecto Sur/ima...
2,dev3-ficheros-es-003,es,ficheros,debian-12,apt,[],Usa rsync para copiar /home/ines/borradores/contrato.odt a /mnt/respaldo/con...,command,[cp -p -- /home/ines/borradores/contrato.odt /mnt/respaldo/contrato.odt]
3,dev3-ficheros-es-004,es,ficheros,ubuntu-22.04,apt,[sudo],Borra definitivamente todos los ficheros regulares cuyo nombre termine en .b...,command,[sudo find /srv/contabilidad -type f -name '*.bak' -delete]
4,dev3-ficheros-es-005,es,ficheros,ubuntu-24.04,apt,"[curl, git, jq, ssh]","Lista las entradas de /srv/entregas, sin incluir las ocultas, de modificació...",command,[ls -lt -- /srv/entregas]


### Transformación 3 — Filtrar registros

Se conservan solo las peticiones con `expected_type == "command"`, es decir, las que deben
responderse con un comando bash. Se descartan las que el asistente debe rechazar (`refuse`, por
estar fuera de ámbito) y las ambiguas (`unsure`).

In [9]:
print("Antes del filtro:")
print(df_selected["expected_type"].value_counts())

df_filtered = df_selected[df_selected["expected_type"] == "command"].reset_index(drop=True)

print()
print(f"Filas antes: {len(df_selected)} | después: {len(df_filtered)}")
print("Tipos restantes:", df_filtered["expected_type"].unique())
df_filtered.head()

Antes del filtro:
expected_type
command    339
refuse     101
unsure      31
Name: count, dtype: int64

Filas antes: 471 | después: 339
Tipos restantes: <ArrowStringArray>
['command']
Length: 1, dtype: str


,id,lang,category,distro,pm,tools,request,expected_type,reference_commands
0,dev3-ficheros-es-001,es,ficheros,ubuntu-22.04,apt,"[batcat, fdfind, rg, tree]","Enséñame en forma de árbol todos los directorios de /srv/musica, incluidos l...",command,[tree -a -d /srv/musica]
1,dev3-ficheros-es-002,es,ficheros,ubuntu-24.04,apt,[],Necesito crear /home/ana/Proyecto Sur/documentos y /home/ana/Proyecto Sur/im...,command,[mkdir -p -- '/home/ana/Proyecto Sur/documentos' '/home/ana/Proyecto Sur/ima...
2,dev3-ficheros-es-003,es,ficheros,debian-12,apt,[],Usa rsync para copiar /home/ines/borradores/contrato.odt a /mnt/respaldo/con...,command,[cp -p -- /home/ines/borradores/contrato.odt /mnt/respaldo/contrato.odt]
3,dev3-ficheros-es-004,es,ficheros,ubuntu-22.04,apt,[sudo],Borra definitivamente todos los ficheros regulares cuyo nombre termine en .b...,command,[sudo find /srv/contabilidad -type f -name '*.bak' -delete]
4,dev3-ficheros-es-005,es,ficheros,ubuntu-24.04,apt,"[curl, git, jq, ssh]","Lista las entradas de /srv/entregas, sin incluir las ocultas, de modificació...",command,[ls -lt -- /srv/entregas]


### Transformación 4 — Crear columnas derivadas

Se crean dos variables numéricas calculadas:

- `request_length`: cantidad de caracteres de la petición. Aproxima qué tan detallada es.
- `n_tools`: cantidad de herramientas instaladas en el entorno (longitud de la lista `tools`).

In [10]:
df_final = df_filtered.copy()
df_final["request_length"] = df_final["request"].str.len()
df_final["n_tools"] = df_final["tools"].apply(len)

df_final[["request", "request_length", "tools", "n_tools"]].head()

,request,request_length,tools,n_tools
0,"Enséñame en forma de árbol todos los directorios de /srv/musica, incluidos l...",114,"[batcat, fdfind, rg, tree]",4
1,Necesito crear /home/ana/Proyecto Sur/documentos y /home/ana/Proyecto Sur/im...,128,[],0
2,Usa rsync para copiar /home/ines/borradores/contrato.odt a /mnt/respaldo/con...,174,[],0
3,Borra definitivamente todos los ficheros regulares cuyo nombre termine en .b...,151,[sudo],1
4,"Lista las entradas de /srv/entregas, sin incluir las ocultas, de modificació...",144,"[curl, git, jq, ssh]",4


In [11]:
# Longitud media de la petición por categoría
df_final.groupby("category")["request_length"].mean().round(1).sort_values(ascending=False)

category
ficheros      134.5
texto         132.0
discos        120.3
git           114.8
compresion    111.0
docker        108.4
red           106.6
permisos      104.0
paquetes      103.9
misc           97.4
procesos       91.2
sistema        76.3
Name: request_length, dtype: float64

## 5. DataFrame final

`df_final` es el resultado de las cuatro transformaciones y es el que se envía a
`DatasetSummarizer`.

In [12]:
print("Dimensiones de df_final:", df_final.shape)
print("Columnas:", list(df_final.columns))
df_final.info()

Dimensiones de df_final: (339, 11)
Columnas: ['id', 'lang', 'category', 'distro', 'pm', 'tools', 'request', 'expected_type', 'reference_commands', 'request_length', 'n_tools']
<class 'pandas.DataFrame'>
RangeIndex: 339 entries, 0 to 338
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   id                  339 non-null    str   
 1   lang                339 non-null    str   
 2   category            339 non-null    str   
 3   distro              339 non-null    str   
 4   pm                  339 non-null    str   
 5   tools               339 non-null    object
 6   request             339 non-null    str   
 7   expected_type       339 non-null    str   
 8   reference_commands  339 non-null    object
 9   request_length      339 non-null    int64 
 10  n_tools             339 non-null    int64 
dtypes: int64(2), object(2), str(7)
memory usage: 81.6+ KB


In [13]:
df_final.describe()

,request_length,n_tools
count,339.000000,339.000000
mean,109.946903,2.592920
std,31.711581,1.823436
min,43.000000,0.000000
25%,87.000000,1.000000
50%,110.000000,3.000000
75%,129.000000,4.000000
max,199.000000,9.000000


## 6. Generación del resumen con `DatasetSummarizer`

In [14]:
OUTPUT_PATH = PROJECT_ROOT / "output" / "resumen_dataset.txt"

summarizer = DatasetSummarizer(df_final, title="Resumen del dataset dale-bash-test-v2 transformado")
saved_path = summarizer.save(OUTPUT_PATH)

print("Resumen guardado en:", saved_path)

Resumen guardado en:

 C:\Proyectos\MachineLearningIU\S10_Unidad1_Evidencia1_Carga_y_Creacion_de_Dataset\output\resumen_dataset.txt


## 7. Comprobación del archivo generado

In [15]:
print("¿Existe el archivo?", saved_path.exists())
print("Tamaño (bytes):", saved_path.stat().st_size)

summary_text = saved_path.read_text(encoding="utf-8")

# El resumen debe corresponder al DataFrame FINAL, no al original
assert f"Cantidad de filas:    {len(df_final)}" in summary_text
assert f"Cantidad de columnas: {df_final.shape[1]}" in summary_text
assert "request_length" in summary_text and "n_tools" in summary_text
print("Verificado: el resumen corresponde a df_final",
      f"({df_final.shape[0]} filas y {df_final.shape[1]} columnas; original: {df.shape}).")

¿Existe el archivo? True
Tamaño (bytes): 3723
Verificado: el resumen corresponde a df_final (339 filas y 11 columnas; original: (471, 8)).


## 8. Contenido del resumen

In [16]:
print(summary_text)

RESUMEN DEL DATASET DALE-BASH-TEST-V2 TRANSFORMADO

Cantidad de filas:    339
Cantidad de columnas: 11

COLUMNAS Y TIPOS DE DATO
----------------------------------------------------------------------
  id                   str
  lang                 str
  category             str
  distro               str
  pm                   str
  tools                object
  request              str
  expected_type        str
  reference_commands   object
  request_length       int64
  n_tools              int64

VALORES NULOS POR COLUMNA
----------------------------------------------------------------------
  id                   0
  lang                 0
  category             0
  distro               0
  pm                   0
  tools                0
  request              0
  expected_type        0
  reference_commands   0
  request_length       0
  n_tools              0

ESTADÍSTICAS DESCRIPTIVAS (columnas numéricas)
----------------------------------------------------------------------
 

## 9. Conclusiones

| # | Transformación | Operación de pandas | Efecto |
|---|----------------|--------------------|--------|
| 1 | Expandir columna anidada | `json_normalize` + `concat` | `context` → `os`, `distro`, `pm`, `tools` |
| 2 | Seleccionar columnas | selección por lista | Se descartan `os` (constante) y `notes` |
| 3 | Filtrar registros | máscara booleana | Solo peticiones `expected_type == "command"` |
| 4 | Columnas derivadas | `str.len` y `apply(len)` | Nuevas variables `request_length` y `n_tools` |

El flujo completo funcionó: los datos se obtuvieron desde Hugging Face con `HFDatasetLoader`,
se transformaron en el Notebook y `DatasetSummarizer` generó el resumen del DataFrame final.